# Phase 2 · Step 5 — Re-ranker on top of hybrid results

Runs hybrid search (Step 4) with a wider net (top 4 instead of top 2),
then re-scores those candidates with a cross-encoder re-ranker and keeps
the real top 2. See `learning/04_rag_pipeline/05_reranker.md` for the why
behind these choices.

## Reload + re-chunk

Same loading/chunking code as the earlier notebooks.

In [1]:
from generate_sample_pdf import generate
from llama_index.core.node_parser import SentenceSplitter
from llama_index.readers.file import PDFReader

pdf_path = generate()
documents = PDFReader().load_data(file=pdf_path)
splitter = SentenceSplitter(chunk_size=512, chunk_overlap=50)
nodes = splitter.get_nodes_from_documents(documents)

print(f"Produced {len(nodes)} chunks from {len(documents)} pages")

Produced 4 chunks from 4 pages


## Recreate the collection + upsert

Same recreate-with-two-named-vectors approach as Step 4 — no change to how
the collection is built, only to what happens after search returns.

In [2]:
from pathlib import Path

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, Modifier, SparseVectorParams, VectorParams

COLLECTION_NAME = "papermind_chunks"
STORAGE_PATH = Path("data/qdrant")
DENSE_MODEL = "BAAI/bge-small-en-v1.5"
SPARSE_MODEL = "Qdrant/bm25"
RERANK_MODEL = "Xenova/ms-marco-MiniLM-L-6-v2"

client = QdrantClient(path=str(STORAGE_PATH))

if client.collection_exists(COLLECTION_NAME):
    client.delete_collection(COLLECTION_NAME)

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config={"dense": VectorParams(size=384, distance=Distance.COSINE)},
    sparse_vectors_config={"bm25": SparseVectorParams(modifier=Modifier.IDF)},
)

print(f"Collection exists: {client.collection_exists(COLLECTION_NAME)}")

Collection exists: True


In [3]:
from fastembed import SparseTextEmbedding, TextEmbedding

texts = [node.get_content() for node in nodes]

dense_embedder = TextEmbedding(model_name=DENSE_MODEL)
sparse_embedder = SparseTextEmbedding(model_name=SPARSE_MODEL)

dense_vectors = list(dense_embedder.embed(texts))
sparse_vectors = list(sparse_embedder.embed(texts))

print(f"Dense: {len(dense_vectors)} vectors, dim={len(dense_vectors[0])}")
print(f"Sparse: {len(sparse_vectors)} vectors, e.g. {len(sparse_vectors[0].indices)} non-zero terms")

Dense: 4 vectors, dim=384
Sparse: 4 vectors, e.g. 19 non-zero terms


In [4]:
from uuid import NAMESPACE_URL, uuid5

from qdrant_client.models import PointStruct, SparseVector

page_counters: dict[str, int] = {}
points = []

for node, dense_vector, sparse_vector in zip(nodes, dense_vectors, sparse_vectors):
    page_label = node.metadata["page_label"]
    file_name = node.metadata["file_name"]
    chunk_index = page_counters.get(page_label, 0)
    page_counters[page_label] = chunk_index + 1

    point_id = str(uuid5(NAMESPACE_URL, f"{file_name}:{page_label}:{chunk_index}"))
    points.append(
        PointStruct(
            id=point_id,
            vector={
                "dense": dense_vector.tolist(),
                "bm25": SparseVector(
                    indices=sparse_vector.indices.tolist(),
                    values=sparse_vector.values.tolist(),
                ),
            },
            payload={
                "text": node.get_content(),
                "page_label": page_label,
                "file_name": file_name,
                "chunk_index": chunk_index,
            },
        )
    )

client.upsert(collection_name=COLLECTION_NAME, points=points)

info = client.get_collection(COLLECTION_NAME)
print("Points count:", info.points_count)
assert info.points_count == len(nodes)

Points count: 4


## Hybrid search — over-fetch

Same as Step 4's search, but `limit=4` (all 4 sample chunks) instead of 2 —
the wider net the re-ranker needs something to narrow down.

In [5]:
from qdrant_client.models import FusionQuery, Fusion, Prefetch

question = "How is citation accuracy measured in this paper?"

query_dense = next(dense_embedder.embed([question])).tolist()
query_sparse = next(sparse_embedder.embed([question]))

hybrid_response = client.query_points(
    collection_name=COLLECTION_NAME,
    prefetch=[
        Prefetch(query=query_dense, using="dense", limit=10),
        Prefetch(
            query=SparseVector(
                indices=query_sparse.indices.tolist(),
                values=query_sparse.values.tolist(),
            ),
            using="bm25",
            limit=10,
        ),
    ],
    query=FusionQuery(fusion=Fusion.RRF),
    limit=4,
    with_payload=True,
)

candidates = hybrid_response.points
print(f"Hybrid search returned {len(candidates)} candidates:")
for c in candidates:
    print(f"  page {c.payload['page_label']} (rrf score={c.score:.3f})")

Hybrid search returned 4 candidates:
  page 4 (rrf score=0.833)
  page 1 (rrf score=0.833)
  page 3 (rrf score=0.250)
  page 2 (rrf score=0.200)


## Re-rank

`TextCrossEncoder.rerank` takes the question once and a list of candidate
texts, and returns one relevance score per candidate — higher is more
relevant. We pair each score back up with its original candidate, then
sort.

In [6]:
import logging
import os

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("huggingface_hub").setLevel(logging.WARNING)

from fastembed.rerank.cross_encoder import TextCrossEncoder

reranker = TextCrossEncoder(model_name=RERANK_MODEL)

candidate_texts = [c.payload["text"] for c in candidates]
rerank_scores = list(reranker.rerank(question, candidate_texts))

scored = sorted(zip(candidates, rerank_scores), key=lambda pair: pair[1], reverse=True)

TOP_K = 2
final_results = scored[:TOP_K]

for candidate, rerank_score in final_results:
    print(f"--- page {candidate.payload['page_label']} (rerank score={rerank_score:.3f}) ---")
    print(candidate.payload["text"])
    print()

--- page 1 (rerank score=2.993) ---
Abstract
This paper studies retrieval-augmented generation for question answering over research
papers. We show that combining dense retrieval with a re-ranking stage improves citation
accuracy compared to retrieval alone.

--- page 4 (rerank score=-3.928) ---
3. Results
Across a held-out set of research papers, grounded answers with page citations were rated
as more trustworthy than ungrounded answers by human reviewers, even when the two answers
conveyed the same information.



## Sanity check

Same lenient check as Steps 3 and 4: at least one of the final results
should actually mention citations. (The re-ranker's own order is discussed
below, in "What we did" in the learning doc — it doesn't match what the
RRF-only order from Step 4 put first.)

In [7]:
pages_returned = [c.payload["page_label"] for c, _ in final_results]
print("Pages returned:", pages_returned)
assert any("citation" in c.payload["text"].lower() for c, _ in final_results)
print("A relevant (citation-mentioning) chunk is in the re-ranked results.")

Pages returned: ['1', '4']
A relevant (citation-mentioning) chunk is in the re-ranked results.


## Next

Phase 2 is now complete end to end: ingest -> embed -> store -> hybrid
search -> re-rank. Phase 3 adds the LLM layer on top of this final
candidate list.